# All Databases Session — source/ and app/ Running Session

Single notebook for a running session across all `source/db-1..db-16` and `app/` output.

**Docker PostgreSQL**: The notebook checks if containers are running (ports 5436–5451) and starts them automatically via `docker start` or `docker compose up -d` when needed. Run `run_docker_qa()` to load schema+data if queries fail.

## 1. Setup

In [26]:
%pip install pyyaml
%pip install psycopg2

import sys
import subprocess
from pathlib import Path
import psycopg2 as psycopg2

ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()
if "notebooks" not in str(ROOT):
    ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path("/Users/machine/Documents/AQ/db")
ROOT = ROOT.resolve()
SOURCE = ROOT / "source"
SCRIPTS = ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS))

DB_PORTS_START = 5436
DB_NUMS = list(range(1, 17))

def get_pg_port(db_num: int) -> int:
    return DB_PORTS_START + db_num - 1

print(f"ROOT: {ROOT}")
print(f"SOURCE: {SOURCE}")
print(f"Ports: {get_pg_port(1)}–{get_pg_port(16)}")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
ROOT: /Users/machine/Documents/AQ/db
SOURCE: /Users/machine/Documents/AQ/db/source
Ports: 5436–5451


## 2. Docker Runtime Check

In [27]:
def check_docker_running() -> bool:
    """Check if Docker daemon is running."""
    try:
        r = subprocess.run(["docker", "info"], capture_output=True, timeout=5)
        return r.returncode == 0
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return False

def check_postgres_containers() -> dict:
    """Check which postgres-db-N containers are running."""
    try:
        r = subprocess.run(
            ["docker", "ps", "--format", "{{.Names}}"],
            capture_output=True, text=True, timeout=5
        )
        if r.returncode != 0:
            return {}
        names = [n.strip() for n in r.stdout.splitlines() if n.strip()]
        return {n: True for n in names if "postgres-db-" in n}
    except Exception:
        return {}

def ensure_postgres_containers() -> bool:
    """Start PostgreSQL containers if not running. Returns True if Docker OK."""
    import time
    if not check_docker_running():
        return False
    containers = check_postgres_containers()
    running = [k for k in containers if containers.get(k)]
    if len(running) >= 16:
        return True
    compose_file = ROOT / "docker" / "docker-compose.hardened.yml"
    print("Starting PostgreSQL containers...")
    print("  Trying docker start (existing containers)...")
    for i in range(1, 17):
        subprocess.run(["docker", "start", f"postgres-db-{i}"], capture_output=True, timeout=5)
    time.sleep(3)
    containers = check_postgres_containers()
    running = [k for k in containers if containers.get(k)]
    if len(running) < 16 and compose_file.exists():
        print("  Some missing. Running docker compose up -d...")
        r2 = subprocess.run(
            ["docker", "compose", "-f", str(compose_file), "up", "-d"],
            cwd=str(ROOT), capture_output=True, text=True, timeout=120
        )
        if r2.returncode != 0 and r2.stderr:
            print(f"  compose stderr: {r2.stderr[:150]}")
        time.sleep(5)
    containers = check_postgres_containers()
    running = [k for k in containers if containers.get(k)]
    print(f"  PostgreSQL containers: {len(running)}/16")
    return len(running) >= 1

docker_ok = check_docker_running()
containers = check_postgres_containers() if docker_ok else {}
running = [k for k in containers if containers.get(k)] if docker_ok else []

if not docker_ok:
    print("⚠️  Docker is NOT running. Start Docker Desktop first.")
else:
    if len(running) < 16:
        ensure_postgres_containers()
    else:
        print(f"✓ Docker running. PostgreSQL containers: {len(running)}/16")

✓ Docker running. PostgreSQL containers: 16/16


## 3. Source Inventory

In [28]:
from db_paths import get_queries_dir, get_data_dir

def source_inventory() -> list:
    rows = []
    for n in DB_NUMS:
        db_dir = SOURCE / f"db-{n}"
        if not db_dir.exists():
            rows.append({"db": f"db-{n}", "queries_json": False, "header": False, "schema": False, "data": False, "app_populated": False})
            continue
        qdir = get_queries_dir(db_dir)
        ddir = get_data_dir(db_dir)
        app_dir = db_dir / "app"
        rows.append({
            "db": f"db-{n}",
            "queries_json": (qdir / "queries.json").exists(),
            "header": (db_dir / "queries_header.yaml").exists() or (db_dir / "queries_header.json").exists(),
            "schema": (ddir / "schema.sql").exists() or (ddir / "schema_postgresql.sql").exists(),
            "data": (ddir / "data.sql").exists(),
            "app_populated": app_dir.exists() and (app_dir / "DATABASE").exists(),
        })
    return rows

try:
    import pandas as pd
    df = pd.DataFrame(source_inventory())
    try:
        from IPython.display import display
        display(df)
    except ImportError:
        print(df.to_string())
except ImportError:
    for r in source_inventory():
        print(r)

{'db': 'db-1', 'queries_json': True, 'header': True, 'schema': True, 'data': True, 'app_populated': True}
{'db': 'db-2', 'queries_json': True, 'header': False, 'schema': True, 'data': True, 'app_populated': True}
{'db': 'db-3', 'queries_json': True, 'header': False, 'schema': True, 'data': True, 'app_populated': True}
{'db': 'db-4', 'queries_json': True, 'header': False, 'schema': True, 'data': True, 'app_populated': True}
{'db': 'db-5', 'queries_json': True, 'header': False, 'schema': True, 'data': True, 'app_populated': True}
{'db': 'db-6', 'queries_json': True, 'header': False, 'schema': True, 'data': True, 'app_populated': True}
{'db': 'db-7', 'queries_json': True, 'header': False, 'schema': True, 'data': True, 'app_populated': True}
{'db': 'db-8', 'queries_json': True, 'header': False, 'schema': True, 'data': True, 'app_populated': True}
{'db': 'db-9', 'queries_json': True, 'header': False, 'schema': True, 'data': True, 'app_populated': True}
{'db': 'db-10', 'queries_json': True, 

## 4. Run Source Checks

In [29]:
try:
    import yaml
except ImportError:
    print("⚠️  PyYAML required for db-1 queries_header.yaml. Run: pip install pyyaml")
    raise

from source_material_checks import check_db

results = [check_db(n) for n in DB_NUMS]
for r in results:
    status = "✓ PASS" if r["pass"] else "✗ FAIL"
    print(f"  {r['db_id']}: {status}")
    for e in r.get("errors", []):
        print(f"    ERROR: {e}")
    for w in r.get("warnings", [])[:2]:
        print(f"    WARN: {w}")

  db-1: ✓ PASS
  db-2: ✓ PASS
  db-3: ✓ PASS
  db-4: ✓ PASS
  db-5: ✓ PASS
  db-6: ✓ PASS
  db-7: ✓ PASS
  db-8: ✓ PASS
  db-9: ✓ PASS
  db-10: ✓ PASS
  db-11: ✓ PASS
  db-12: ✓ PASS
  db-13: ✓ PASS
  db-14: ✓ PASS
  db-15: ✓ PASS
  db-16: ✓ PASS


## 5. Docker PostgreSQL Status

In [30]:
def try_connect(db_num: int):
    """Try connecting to PostgreSQL for db-N. Returns (connected, message)."""
    try:
        import psycopg2
        port = get_pg_port(db_num)
        conn = psycopg2.connect(
            host="localhost", port=port, user="postgres", password="postgres", dbname=f"db{db_num}", connect_timeout=2
        )
        cur = conn.cursor()
        cur.execute("SELECT count(*) FROM information_schema.tables WHERE table_schema = 'public'")
        cnt = cur.fetchone()[0]
        conn.close()
        return True, f"{cnt} tables"
    except Exception as e:
        return False, str(e)[:60]

if docker_ok:
    pg_status = []
    for n in DB_NUMS:
        ok, msg = try_connect(n)
        pg_status.append({"db": f"db-{n}", "port": get_pg_port(n), "connected": ok, "schema_info": msg})
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(pg_status))
    except ImportError:
        for r in pg_status:
            print(r)
else:
    print("Docker not running — skip PostgreSQL status.")

{'db': 'db-1', 'port': 5436, 'connected': True, 'schema_info': '15 tables'}
{'db': 'db-2', 'port': 5437, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-3', 'port': 5438, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-4', 'port': 5439, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-5', 'port': 5440, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-6', 'port': 5441, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-7', 'port': 5442, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-8', 'port': 5443, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-9', 'port': 5444, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-10', 'port': 5445, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-11', 'port': 5446, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-12', 'port': 5447, 'connected': True, 'schema_info': '3 tables'}
{'db': 'db-13', 'port': 5448, 'connected': True, 'schema_info': '14 tables'}
{'db': 'db-14', 'po

## 6. Sync Actions

In [31]:
# Run populate_app_trifecta and resync_client_db for all dbs
def run_populate_app():
    subprocess.run([sys.executable, str(SCRIPTS / "populate_app_trifecta.py"), "-a"], cwd=ROOT, check=False)

def run_resync_client():
    subprocess.run([sys.executable, str(SCRIPTS / "resync_client_db.py")], cwd=ROOT, check=False)

def run_docker_qa():
    subprocess.run(["bash", str(ROOT / "scripts" / "docker_postgres_qa.sh"), "-a"], cwd=ROOT, check=False)

print("Call run_populate_app(), run_resync_client(), or run_docker_qa() to sync.")

Call run_populate_app(), run_resync_client(), or run_docker_qa() to sync.


## 7. Query Execution Sample

In [32]:
import json

def run_query_1(db_num: int):
    """Run Query 1 from queries.json against Docker PostgreSQL."""
    import re
    qdir = get_queries_dir(SOURCE / f"db-{db_num}")
    qj = qdir / "queries.json"
    if not qj.exists():
        return None, "no queries.json"
    data = json.loads(qj.read_text())
    queries = data.get("queries") if isinstance(data, dict) else (data if isinstance(data, list) else [])
    if not queries or not isinstance(queries, list):
        return None, "no queries"
    first = queries[0] if isinstance(queries[0], dict) else {}
    sql = first.get("SQL") or first.get("sql", "")
    if not sql:
        return None, "no SQL"
    sql_clean = re.sub(r"\s+LIMIT\s+\d+\s*;?\s*$", "", sql.strip(), flags=re.I).rstrip(";").strip()
    sql_limited = sql_clean + " LIMIT 5"
    try:
        import psycopg2
        port = get_pg_port(db_num)
        conn = psycopg2.connect(host="localhost", port=port, user="postgres", password="postgres", dbname=f"db{db_num}", connect_timeout=2)
        cur = conn.cursor()
        cur.execute(sql_limited)
        rows = cur.fetchall()
        conn.close()
        return rows, f"{len(rows)} rows"
    except Exception as e:
        return None, str(e)[:80]

def verify_query_paths():
    """Verify queries.json path and structure for each db (no DB connection)."""
    for n in [1, 2, 3]:
        qdir = get_queries_dir(SOURCE / f"db-{n}")
        qj = qdir / "queries.json"
        ok = qj.exists()
        msg = f"db-{n}: {qdir.relative_to(ROOT) if ok else 'missing'}"
        if ok:
            try:
                data = json.loads(qj.read_text())
                qs = data.get("queries") if isinstance(data, dict) else (data if isinstance(data, list) else [])
                first = qs[0] if qs and isinstance(qs[0], dict) else {}
                has_sql = bool(first.get("SQL") or first.get("sql"))
                msg += f" | queries={len(qs) if qs else 0} | sql={'✓' if has_sql else '✗'}"
            except Exception as e:
                msg += f" | error: {str(e)[:40]}"
        print(msg)

if docker_ok:
    verify_query_paths()
    print("")
    for n in [1, 2, 3]:
        rows, msg = run_query_1(n)
        print(f"db-{n} Query 1: {msg}")
        if rows:
            print(rows[:2])
else:
    print("Docker not running — skip query execution.")

db-1: source/db-1/app/QUERIES | queries=30 | sql=✓
db-2: source/db-2/app/QUERIES | queries=30 | sql=✓
db-3: source/db-3/app/QUERIES | queries=30 | sql=✓

db-1 Query 1: 0 rows
db-2 Query 1: relation "phppos_sales" does not exist
LINE 6:     FROM phppos_sales
           
db-3 Query 1: relation "orders_order" does not exist
LINE 9:     FROM orders_order
           


## 8. Change Propagation (File-Based CDC)

In [10]:
# Optional: start watch_source_and_sync in background
# Run: python3 scripts/watch_source_and_sync.py
# This watches source/ for changes to queries.json, schema.sql, etc., then runs checks → populate → resync
print("To enable file-based CDC, run in a terminal: python3 scripts/watch_source_and_sync.py")

To enable file-based CDC, run in a terminal: python3 scripts/watch_source_and_sync.py
